# تقطیر واقعی FFN کوئن به MoE با top-2

این نوت‌بوک خروجی واقعی FFN یک لایه از `Qwen/Qwen3.5-2B` را با hook جمع می‌کند و Router و ۸ expert کانال‌شاردشده را با MSE آموزش می‌دهد. این **فقط آزمایش یک لایه** است؛ مدل کامل تبدیل یا آموزش نمی‌شود.

**هزینه و حریم منابع:** هیچ GPU یا سرویس پولی خودکار فعال نمی‌شود. اگر محیط رایگان GPU در دسترس نیست، CPU را انتخاب کنید؛ اجرای CPU ممکن است کند باشد. بارگذاری مدل به چند گیگابایت RAM نیاز دارد.

برای نتیجهٔ معنادار، یک فایل UTF-8 از متن‌های فارسی/متنوع خودتان آماده کنید، هر نمونه در یک خط و حداقل ۴ خط. فایل نمونهٔ بسیار کوچک فقط برای smoke test است و کیفیت را ارزیابی نمی‌کند.

In [ ]:
%pip -q install -U torch transformers accelerate safetensors


In [ ]:
import os, torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('CPU mode; no paid runtime will be started automatically.')


In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload your UTF-8 .txt corpus; one text sample per line
assert uploaded, 'Upload a UTF-8 text file first.'
name = next(iter(uploaded))
TEXT_PATH = '/content/fa_train.txt'
with open(TEXT_PATH, 'wb') as f:
    f.write(uploaded[name])
print('Corpus saved:', TEXT_PATH, 'bytes:', os.path.getsize(TEXT_PATH))


In [ ]:
!git clone --depth 1 https://github.com/samanramezani1377-hub/Qwen-MoE-0.4B-.git /content/Qwen-MoE-0.4B-
%cd /content/Qwen-MoE-0.4B-
!python scripts/distill_real_qwen_ffn.py --text-file "$TEXT_PATH" --layer 0 --samples 256 --max-length 128 --num-experts 8 --top-k 2 --epochs 3 --batch-size 4 --device auto --output /content/layer0_top2.pt


## نتیجه و محدودیت‌ها

عدد `eval_mse` خطای بازسازی خروجی FFN در توکن‌های نگه‌داشته‌شده است؛ به‌تنهایی معادل کیفیت زبانی یا استدلال نیست. این آزمایش هنوز perplexity یا پاسخ‌گویی انتها‌به‌انتها را نمی‌سنجد. برای آزمایش بعدی باید با متن بیشتر، لایه‌های مختلف، معیارهای ثابت و در نهایت ارزیابی مدل کامل مقایسه شود.

فایل checkpoint در `/content/layer0_top2.pt` ذخیره می‌شود و با پایان جلسهٔ Colab ممکن است از بین برود؛ برای نگهداری، آن را دانلود کنید.

In [ ]:
from google.colab import files
if os.path.exists('/content/layer0_top2.pt'):
    files.download('/content/layer0_top2.pt')
else:
    print('No checkpoint yet; run the training cell successfully first.')
